# GROUP 10: University of Vienna – Business Intelligence 1

In [1]:
import pandas as pd
import numpy as np

In [2]:
#https://sqlpey.com/python/top-3-methods-to-open-csv-files-in-data-folders-using-pandas-with-relative-paths/
income_data = pd.read_csv("../data/raw/vie-bez-biz-ecn-inc-sex-2002f.csv", sep=";", skiprows=1)
unemployment_data = pd.read_csv("../data/raw/vie-bez-biz-emp-sex-uep-2002f.csv", sep=";", skiprows=1)

In [3]:
income_data.head()

,NUTS,DISTRICT_CODE,SUB_DISTRICT_CODE,REF_YEAR,REF_DATE,INC_TOT_VALUE,INC_MAL_VALUE,INC_FEM_VALUE,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,AT13,90000,90000,2002,20021231,18.217,20.709,15.424,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AT13,90100,90100,2002,20021231,25.463,31.961,18.536,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AT13,90200,90200,2002,20021231,16.439,18.301,14.282,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AT13,90300,90300,2002,20021231,18.701,21.444,15.804,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AT13,90400,90400,2002,20021231,20.325,23.641,16.876,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
unemployment_data.head()

,NUTS,DISTRICT_CODE,SUB_DISTRICT_CODE,REF_YEAR,REF_DATE,SEX,UEP_VALUE,UEP_DENSITY,Unnamed: 8,Unnamed: 9
0,AT13,90000,90000,2002,2002,0,74.894,"67,83",NaN,NaN
1,AT13,90100,90100,2002,2002,0,433.000,"35,32",NaN,NaN
2,AT13,90200,90200,2002,2002,0,4.784,"76,98",NaN,NaN
3,AT13,90300,90300,2002,2002,0,3.957,"68,25",NaN,NaN
4,AT13,90400,90400,2002,2002,0,1.140,"55,68",NaN,NaN


## Data Transformation and Integration

### Dim_Sex

In [5]:
dim_sex_data = {
    'id':["0", "1"],
    'code':["m", "f"],
    'name':["Male", "Female"],
}
dim_sex_df = pd.DataFrame(dim_sex_data)
dim_sex_df.to_csv('../data/created/dim_sex_data.csv', sep=';', index=False) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html#pandas.DataFrame.to_csv

### Dim_Time

In [6]:
#Check if both REF_YEAR and REF_DATE have the same values for unemployment_data
unemployment_data["REF_YEAR"].equals(unemployment_data["REF_DATE"]) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.equals.html

True

In [7]:
#Check if both REF_YEAR and REF_DATE have the same year for income_data
income_data["REF_YEAR"].astype(str).equals(income_data["REF_DATE"].astype(str).str[:4]) #https://stackoverflow.com/questions/36505847/substring-of-an-entire-column-in-pandas-dataframe

True

In [8]:
year_min = min(income_data["REF_YEAR"].min(), unemployment_data["REF_YEAR"].min())
year_min

2002

In [9]:
year_max = max(income_data["REF_YEAR"].max(), unemployment_data["REF_YEAR"].max())
year_max

2023

In [10]:
dim_date_data = {
    'id':list(range(0, year_max-year_min+1,1)), #https://stackoverflow.com/questions/40706034/how-to-create-a-list-of-a-range-with-incremental-step
    'year':list(range(year_min,year_max+1,1)) #https://www.w3schools.com/python/ref_func_range.asp
}
dim_date_df = pd.DataFrame(dim_date_data) 
dim_date_df.to_csv('../data/created/dim_date_data.csv', sep=';', index=False) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html#pandas.DataFrame.to_csv

### Dim_District

In [11]:
#Check if both DISTRICT_CODE and SUB_DISTRICT_CODE have the same values for income_data
income_data["DISTRICT_CODE"].equals(income_data["SUB_DISTRICT_CODE"]) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.equals.html

True

In [12]:
#Check if both DISTRICT_CODE and SUB_DISTRICT_CODE have the same values for income_data
unemployment_data["DISTRICT_CODE"].equals(unemployment_data["SUB_DISTRICT_CODE"]) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.equals.html

True

In [13]:
#Check how many states
income_data["NUTS"].unique()

array(['AT13'], dtype=object)

In [14]:
#Check how many districts
districts = income_data["DISTRICT_CODE"].astype(str).str[1:3].unique() #https://stackoverflow.com/questions/36505847/substring-of-an-entire-column-in-pandas-dataframe
districts = list(districts) #https://numpy.org/doc/stable/reference/generated/numpy.ndarray.tolist.html
#remove district 00, since I assume it is the total)
districts.pop(0) #https://www.geeksforgeeks.org/python/python-removing-first-element-of-list/

'00'

In [15]:
dim_district_data = {
    'id':list(range(0, len(districts), 1)), #https://stackoverflow.com/questions/40706034/how-to-create-a-list-of-a-range-with-incremental-step
    'state':[income_data["NUTS"].unique()[0]] * len(districts), #https://stackoverflow.com/questions/3459098/create-list-of-single-item-repeated-n-times
    'district':districts,
    'area_km2':[""] * len(districts)
}
dim_district_df = pd.DataFrame(dim_district_data) 
dim_district_df.to_csv('../data/created/dim_district_data.csv', sep=';', index=False) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html#pandas.DataFrame.to_csv

### Fact_Economic_Indicators

- id (PK)
- avg_net_income
- unemployment_count
- unemployment_percentage
- population
- ditrictID (FK)
- sexID (FK)
- dateID (FK)

#### Integrating unemployment_data

In [16]:
#Drop rows with total value of both genders
fact_economic_indicators = unemployment_data[unemployment_data["SEX"] != 0] #https://www.geeksforgeeks.org/python/how-to-drop-rows-in-dataframe-by-conditions-on-column-values/

#Reset index
fact_economic_indicators = fact_economic_indicators.reset_index(drop=True) #https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html

#Drop rows with total value of all districts
fact_economic_indicators['DISTRICT_CODE'] = fact_economic_indicators['DISTRICT_CODE'].apply(str) #https://stackoverflow.com/questions/17950374/converting-a-column-within-pandas-dataframe-from-int-to-string
fact_economic_indicators = fact_economic_indicators[fact_economic_indicators["DISTRICT_CODE"].str[1:3] != '00'] #https://www.geeksforgeeks.org/python/how-to-drop-rows-in-dataframe-by-conditions-on-column-values/

#Reset index
fact_economic_indicators = fact_economic_indicators.reset_index(drop=True) #https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html
fact_economic_indicators

,NUTS,DISTRICT_CODE,SUB_DISTRICT_CODE,REF_YEAR,REF_DATE,SEX,UEP_VALUE,UEP_DENSITY,Unnamed: 8,Unnamed: 9
0,AT13,90100,90100,2002,2002,1,232.000,"38,03",NaN,NaN
1,AT13,90200,90200,2002,2002,1,2.933,"92,67",NaN,NaN
2,AT13,90300,90300,2002,2002,1,2.403,"83,54",NaN,NaN
3,AT13,90400,90400,2002,2002,1,668.000,"66,57",NaN,NaN
4,AT13,90500,90500,2002,2002,1,1.734,"95,56",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1007,AT13,91900,91900,2023,2023,2,1.621,"64,04",NaN,NaN
1008,AT13,92000,92000,2023,2023,2,3.394,"115,03",NaN,NaN
1009,AT13,92100,92100,2023,2023,2,6.503,"103,66",NaN,NaN
1010,AT13,92200,92200,2023,2023,2,5.993,"78,59",NaN,NaN


In [18]:
#Drop irrelevant columns
fact_economic_indicators = fact_economic_indicators.drop(['NUTS','SUB_DISTRICT_CODE', 'REF_DATE', 'Unnamed: 8', 'Unnamed: 9'], axis=1) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html
fact_economic_indicators

,DISTRICT_CODE,REF_YEAR,SEX,UEP_VALUE,UEP_DENSITY
0,90100,2002,1,232.000,"38,03"
1,90200,2002,1,2.933,"92,67"
2,90300,2002,1,2.403,"83,54"
3,90400,2002,1,668.000,"66,57"
4,90500,2002,1,1.734,"95,56"
...,...,...,...,...,...
1007,91900,2023,2,1.621,"64,04"
1008,92000,2023,2,3.394,"115,03"
1009,92100,2023,2,6.503,"103,66"
1010,92200,2023,2,5.993,"78,59"


In [19]:
#Rename columns
fact_economic_indicators = fact_economic_indicators.rename(columns={ #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html
    "DISTRICT_CODE": "districtID", 
    "REF_YEAR": "dateID",
    "SEX": "sexID",
    "UEP_VALUE": "unemployment_count",
    "UEP_DENSITY": "unemployment_percentage"
})
fact_economic_indicators

,districtID,dateID,sexID,unemployment_count,unemployment_percentage
0,90100,2002,1,232.000,"38,03"
1,90200,2002,1,2.933,"92,67"
2,90300,2002,1,2.403,"83,54"
3,90400,2002,1,668.000,"66,57"
4,90500,2002,1,1.734,"95,56"
...,...,...,...,...,...
1007,91900,2023,2,1.621,"64,04"
1008,92000,2023,2,3.394,"115,03"
1009,92100,2023,2,6.503,"103,66"
1010,92200,2023,2,5.993,"78,59"


In [20]:
#Replace descriptive column values with their IDs
#https://www.geeksforgeeks.org/pandas/different-ways-to-iterate-over-rows-in-pandas-dataframe/
for i, row in fact_economic_indicators.iterrows():
    #dateID
    year = fact_economic_indicators.loc[i,'dateID']
    fact_economic_indicators.loc[i,'dateID'] = int(dim_date_df[dim_date_df["year"] == year]["id"].iloc[0])
    
    #sexID
    fact_economic_indicators.loc[i,'sexID'] = 0 if fact_economic_indicators.loc[i,'sexID'] == 1 else 1 #https://stackoverflow.com/questions/18669836/is-it-possible-to-write-single-line-return-statement-with-if-statement
    
    #districtID
    district = fact_economic_indicators.loc[i,'districtID'][1:3]
    fact_economic_indicators.loc[i,'districtID'] = int(dim_district_df[dim_district_df["district"] == district]["id"].iloc[0])
    
fact_economic_indicators

,districtID,dateID,sexID,unemployment_count,unemployment_percentage
0,0,0,0,232.000,"38,03"
1,1,0,0,2.933,"92,67"
2,2,0,0,2.403,"83,54"
3,3,0,0,668.000,"66,57"
4,4,0,0,1.734,"95,56"
...,...,...,...,...,...
1007,18,21,1,1.621,"64,04"
1008,19,21,1,3.394,"115,03"
1009,20,21,1,6.503,"103,66"
1010,21,21,1,5.993,"78,59"


#### Integrating income_data

In [21]:
#Add missing columns
# https://www.geeksforgeeks.org/pandas/adding-new-column-to-existing-dataframe-in-pandas/
# fact_economic_indicators.insert(0, "id", [""]*len(fact_economic_indicators), True)
# fact_economic_indicators.insert(1, "id", [""]*len(fact_economic_indicators), True)

## Data Cleaning